# AnyCam — 02 · Setup, data check, and first camera simulation (Steps 2 and 4-preview)

**Before running, in the right panel:**
1. **Add Input** → Competitions → `HyperLeaf2024` (accept its rules first on the competition page)
2. **Add Input** → Your Datasets → `anycam-extra-data` (made by notebook 01)
3. **Settings → Internet → On**
4. If your GitHub repo is **private**: Add-ons → Secrets → add `GITHUB_TOKEN` (see the guide)

Then **Run All**. At the end, download the two report files from the Output tab and send them to Claude.

In [ ]:
GITHUB_USER = 'YOUR_GITHUB_USERNAME'   # <-- change this
REPO = 'anycam'

import os, subprocess, glob, pathlib
token = ''
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    pass  # public repo: no token needed
auth = f'{token}@' if token else ''
if not os.path.exists('/kaggle/working/anycam'):
    subprocess.run(['git', 'clone', '-q', f'https://{auth}github.com/{GITHUB_USER}/{REPO}.git', '/kaggle/working/anycam'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/anycam', 'pull', '-q'], check=True)
os.chdir('/kaggle/working/anycam')
print('code ready:', os.listdir('.'))

In [ ]:
!pip -q install pyyaml spectral tifffile h5py
!python -m pytest -q
!python scripts/list_cameras.py

In [ ]:
# GPU check (only matters for training later; turn on Settings -> Accelerator -> GPU T4 x2)
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Where did Kaggle put the data?
import glob
for d in sorted(glob.glob('/kaggle/input/*')):
    print(d)
    for sub in sorted(glob.glob(d + '/*'))[:15]:
        print('   ', sub)

In [ ]:
# Step 2: one data report per input dataset -> /kaggle/working/anycam/reports/
for d in sorted(glob.glob('/kaggle/input/*')):
    name = pathlib.Path(d).name
    print('=' * 30, name)
    subprocess.run(['python', 'scripts/inspect_data.py', d, f'reports/{name}_DATA_REPORT.md'])

In [ ]:
# Step 1 on REAL data: turn one HyperLeaf leaf into drone and phone images
hl = [d for d in glob.glob('/kaggle/input/*') if 'hyperleaf' in d.lower()]
if hl:
    subprocess.run(['python', 'scripts/first_simulation.py', hl[0], 'reports/first_simulation_hyperleaf.png'])
    from IPython.display import Image, display
    display(Image('reports/first_simulation_hyperleaf.png'))
else:
    print('HyperLeaf2024 is not added: Add Input -> Competitions -> HyperLeaf2024')

## Done: send these to Claude
Open the **Output** tab on the right (or the folder `/kaggle/working/anycam/reports/`), download:
- every `*_DATA_REPORT.md` file
- `first_simulation_hyperleaf.png`

and paste or upload them in the chat.